<a href="https://colab.research.google.com/github/MohitKhetan10/Seq2seq_model/blob/main/Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

# ==========================================
# Helper Function: Softmax
# ==========================================
# Converts raw alignment scores into percentages (probabilities).
# Ensures that all values add up to 1 (100% focus).
def softmax(x):
    e_x = np.exp(x - np.max(x))  # stability trick
    return e_x / e_x.sum()

# ==========================================
# 1. THE SETUP
# ==========================================
# Input sentence (English): "I love machine learning"
# Encoder produces hidden states for each word.
encoder_states = np.array([
    [0.2, 0.5, -0.1],  # h1: "I"
    [0.9, 0.1,  0.8],  # h2: "love"
    [-0.3, 0.7, 0.2],  # h3: "machine"
    [0.4, -0.2, 0.6]   # h4: "learning"
])

# Decoder’s current hidden state (its "brain" while predicting)
decoder_state = np.array([0.8, 0.2, 0.7])

print("=== Seq2Seq with Attention Simulation ===")
print("Input (English): I love machine learning")
print("Target (French): J’aime l’apprentissage automatique\n")

# ==========================================
# 2. CALCULATE ATTENTION SCORES (The Search)
# ==========================================
raw_scores = []

print("Step 1: Calculating similarity between Decoder and each Encoder word...")

for idx, h_i in enumerate(encoder_states):
    # Dot product = similarity score
    score = np.dot(decoder_state, h_i)
    raw_scores.append(score)
    print(f"Similarity with '{['I','love','machine','learning'][idx]}': {round(score, 3)}")

print(f"\nRaw Scores: {np.round(raw_scores, 3)}")

# ==========================================
# 3. ATTENTION WEIGHTS (Softmax)
# ==========================================
attention_weights = softmax(raw_scores)

print("\nStep 2: Converting scores into percentages (attention weights)...")
for idx, weight in enumerate(attention_weights):
    print(f"Focus on '{['I','love','machine','learning'][idx]}': {round(weight*100, 1)}%")

print(f"\nAttention Weights (%): {np.round(attention_weights*100, 1)}")

# ==========================================
# 4. BUILD THE DYNAMIC CONTEXT VECTOR
# ==========================================
# Weighted sum of encoder states using attention weights
dynamic_context_vector = np.dot(attention_weights, encoder_states)

print("\nStep 3: Building the Dynamic Context Vector...")
print(f"Dynamic Context Vector (summary of input): {np.round(dynamic_context_vector, 3)}")

# ==========================================
# 5. FINAL PREDICTION INPUT
# ==========================================
# Combine decoder state + context vector
final_input = np.concatenate((decoder_state, dynamic_context_vector))

print("\nStep 4: Final Combined Vector for Prediction...")
print(f"Final Input Vector: {np.round(final_input, 3)}")

# ==========================================
# 6. INTERPRETATION (Layman’s Explanation)
# ==========================================
print("\n=== INTERPRETATION ===")
print("The decoder is trying to translate the first French word.")
print("It looked at all English words and decided:")
print("- 46% focus on 'love'")
print("- 25% focus on 'learning'")
print("- Smaller focus on 'I' and 'machine'")
print("\nThis makes sense: to translate 'I love...',")
print("the model emphasizes 'love' and 'learning'.")
print("So the predicted French word is: 'J’aime'.")
print("\nOn the next steps, attention will shift to 'machine' and 'learning',")
print("producing 'l’apprentissage automatique'.")
print("\n✅ Final Translation: J’aime l’apprentissage automatique")


=== Seq2Seq with Attention Simulation ===
Input (English): I love machine learning
Target (French): J’aime l’apprentissage automatique

Step 1: Calculating similarity between Decoder and each Encoder word...
Similarity with 'I': 0.19
Similarity with 'love': 1.3
Similarity with 'machine': 0.04
Similarity with 'learning': 0.7

Raw Scores: [0.19 1.3  0.04 0.7 ]

Step 2: Converting scores into percentages (attention weights)...
Focus on 'I': 15.2%
Focus on 'love': 46.3%
Focus on 'machine': 13.1%
Focus on 'learning': 25.4%

Attention Weights (%): [15.2 46.3 13.1 25.4]

Step 3: Building the Dynamic Context Vector...
Dynamic Context Vector (summary of input): [0.509 0.164 0.533]

Step 4: Final Combined Vector for Prediction...
Final Input Vector: [0.8   0.2   0.7   0.509 0.164 0.533]

=== INTERPRETATION ===
The decoder is trying to translate the first French word.
It looked at all English words and decided:
- 46% focus on 'love'
- 25% focus on 'learning'
- Smaller focus on 'I' and 'machine'



In [4]:
import numpy as np

# ==========================================
# Helper Function: Softmax
# ==========================================
def softmax(x):
    e_x = np.exp(x - np.max(x))  # stability trick
    return e_x / e_x.sum()

# ==========================================
# 1. THE SETUP
# ==========================================
# Input paragraph (English):
# "Artificial Intelligence is transforming the world.
#  It helps in healthcare, education, and communication.
#  Machine learning allows computers to learn from data
#  and make predictions that improve our daily lives."

# For simplicity, we represent each English word as a 3D vector (encoder states).
# In real systems, these vectors come from trained embeddings.
encoder_states = np.array([
    [0.2, 0.5, -0.1],   # "Artificial"
    [0.9, 0.1,  0.8],   # "Intelligence"
    [-0.3, 0.7, 0.2],   # "is"
    [0.4, -0.2, 0.6],   # "transforming"
    [0.5, 0.3, 0.1],    # "the"
    [0.7, 0.4, 0.9],    # "world"
    [0.6, 0.2, 0.3],    # "healthcare"
    [0.3, 0.8, 0.2],    # "education"
    [0.1, 0.9, 0.4],    # "communication"
    [0.8, 0.2, 0.7],    # "Machine"
    [0.2, 0.6, 0.5],    # "learning"
    [0.9, 0.3, 0.2],    # "data"
    [0.4, 0.7, 0.6],    # "predictions"
    [0.5, 0.5, 0.5]     # "lives"
])

# Decoder initial state (its "brain" before generating words)
decoder_state = np.array([0.8, 0.2, 0.7])

# Target French translation (for demonstration)
french_targets = [
    "L’intelligence artificielle transforme le monde.",
    "Elle aide dans la santé, l’éducation et la communication.",
    "L’apprentissage automatique permet aux ordinateurs d’apprendre des données",
    "et de faire des prédictions qui améliorent notre vie quotidienne."
]

print("=== Seq2Seq with Attention Simulation ===")
print("Input (English Paragraph):")
print("Artificial Intelligence is transforming the world. It helps in healthcare, education, and communication. Machine learning allows computers to learn from data and make predictions that improve our daily lives.\n")
print("Target (French Translation):")
for line in french_targets:
    print(line)
print("\n")

# ==========================================
# 2. DECODING LOOP (Word by Word Simulation)
# ==========================================
for step, target_sentence in enumerate(french_targets):
    print(f"\n==============================")
    print(f"Step {step + 1}: Generating French sentence")
    print("==============================")
    print(f"Target Output: {target_sentence}\n")

    # 1️⃣ Alignment Scores
    raw_scores = np.dot(encoder_states, decoder_state)
    print("Raw Alignment Scores (similarity with each English word):")
    print(np.round(raw_scores, 3))

    # 2️⃣ Attention Weights
    attention_weights = softmax(raw_scores)
    print("\nAttention Weights (% focus on each word):")
    print(np.round(attention_weights * 100, 1))

    # 3️⃣ Dynamic Context Vector
    dynamic_context_vector = np.dot(attention_weights, encoder_states)
    print("\nDynamic Context Vector (summary of input):")
    print(np.round(dynamic_context_vector, 3))

    # 4️⃣ Final Input
    final_input = np.concatenate((decoder_state, dynamic_context_vector))
    print("\nFinal Input Vector for Prediction:")
    print(np.round(final_input, 3))

    # 5️⃣ Update Decoder State (simulate evolution)
    decoder_state = decoder_state + np.random.uniform(-0.1, 0.1, size=decoder_state.shape)
    print("\nUpdated Decoder State for Next Step:")
    print(np.round(decoder_state, 3))

print("\n✅ Final Translation Completed:")
for line in french_targets:
    print(line)


=== Seq2Seq with Attention Simulation ===
Input (English Paragraph):
Artificial Intelligence is transforming the world. It helps in healthcare, education, and communication. Machine learning allows computers to learn from data and make predictions that improve our daily lives.

Target (French Translation):
L’intelligence artificielle transforme le monde.
Elle aide dans la santé, l’éducation et la communication.
L’apprentissage automatique permet aux ordinateurs d’apprendre des données
et de faire des prédictions qui améliorent notre vie quotidienne.



Step 1: Generating French sentence
Target Output: L’intelligence artificielle transforme le monde.

Raw Alignment Scores (similarity with each English word):
[0.19 1.3  0.04 0.7  0.53 1.27 0.73 0.54 0.54 1.17 0.63 0.92 0.88 0.85]

Attention Weights (% focus on each word):
[ 3.9 11.8  3.4  6.5  5.5 11.5  6.7  5.5  5.5 10.4  6.   8.1  7.8  7.5]

Dynamic Context Vector (summary of input):
[0.536 0.385 0.501]

Final Input Vector for Predicti